In [70]:
import pandas as pd 

data = pd.read_csv("student-mat.csv", sep=";")
data.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [71]:
print("Dataset shape:", data.shape)
print("\nColumn names:")
print(data.columns.tolist())
print("\nMissing values:")
print(data.isnull().sum().sum())

Dataset shape: (395, 33)

Column names:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']

Missing values:
0


In [72]:
print(data["G3"].describe())

print("\nFinal grade distribution:")
print(data["G3"].value_counts().sort_index())

count    395.000000
mean      10.415190
std        4.581443
min        0.000000
25%        8.000000
50%       11.000000
75%       14.000000
max       20.000000
Name: G3, dtype: float64

Final grade distribution:
G3
0     38
4      1
5      7
6     15
7      9
8     32
9     28
10    56
11    47
12    31
13    31
14    27
15    33
16    16
17     6
18    12
19     5
20     1
Name: count, dtype: int64


In [73]:
data["low_grade"] = (data["G3"] < 10).astype(int)

print(data["low_grade"].value_counts())

low_grade
0    265
1    130
Name: count, dtype: int64


In [74]:
X = data.drop(columns=["G3", "low_grade"])
y = data["low_grade"]

print("Input features:", X.shape)
print("Target:", y.shape)

Input features: (395, 32)
Target: (395,)


In [75]:
numerical_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical columns:", numerical_cols)
print("\nCategorical columns:", categorical_cols)

Numerical columns: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']

Categorical columns: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


C:\Users\DELL\AppData\Local\Temp\ipykernel_11164\1810531239.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [76]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training set: (316, 32)
Testing set: (79, 32)

Training target distribution:
low_grade
0    212
1    104
Name: count, dtype: int64

Testing target distribution:
low_grade
0    53
1    26
Name: count, dtype: int64


In [77]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [78]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.9367088607594937

Classification report:
              precision    recall  f1-score   support

           0       0.93      0.98      0.95        53
           1       0.96      0.85      0.90        26

    accuracy                           0.94        79
   macro avg       0.94      0.91      0.93        79
weighted avg       0.94      0.94      0.94        79

Confusion matrix:
[[52  1]
 [ 4 22]]
ROC-AUC: 0.9767779390420901


In [79]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score

rf_model = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("F1-score:", f1_score(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))

Random Forest
Accuracy: 0.9113924050632911
Recall: 0.8846153846153846
F1-score: 0.8679245283018868
ROC-AUC: 0.9749637155297532


In [80]:
try:
    import xgboost
    print("XGBoost version:", xgboost.__version__)
except ImportError:
    print("XGBoost is not installed.")

XGBoost version: 3.4.1


In [81]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [82]:
import xgboost

print(xgboost.__version__)

3.4.1


In [83]:
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.pipeline import Pipeline

xgb_model = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("classifier", XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss"
    ))
])

xgb_model.fit(X_train, y_train)

print("XGBoost trained successfully!")

XGBoost trained successfully!


In [84]:
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, roc_auc_score
)

xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost")
print("Accuracy:", accuracy_score(y_test, xgb_pred))
print("Recall:", recall_score(y_test, xgb_pred))
print("F1-score:", f1_score(y_test, xgb_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb_prob))

XGBoost
Accuracy: 0.9113924050632911
Recall: 0.8461538461538461
F1-score: 0.8627450980392157
ROC-AUC: 0.974600870827286


In [85]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, xgb_pred)
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, rf_pred),
        recall_score(y_test, xgb_pred)
    ],
    "F1-score": [
        f1_score(y_test, y_pred),
        f1_score(y_test, rf_pred),
        f1_score(y_test, xgb_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob),
        roc_auc_score(y_test, rf_prob),
        roc_auc_score(y_test, xgb_prob)
    ]
})

comparison.round(3)

,Model,Accuracy,Recall,F1-score,ROC-AUC
0,Logistic Regression,0.937,0.846,0.898,0.977
1,Random Forest,0.911,0.885,0.868,0.975
2,XGBoost,0.911,0.846,0.863,0.975


In [86]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Logistic Regression": model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

scoring = ["accuracy", "recall", "f1", "roc_auc"]

for name, estimator in models.items():
    results = cross_validate(
        estimator, X_train, y_train,
        cv=cv, scoring=scoring
    )

    print(f"\n{name}")
    for metric in scoring:
        scores = results[f"test_{metric}"]
        print(f"{metric}: {scores.mean():.3f} (+/- {scores.std():.3f})")


Logistic Regression
accuracy: 0.905 (+/- 0.042)
recall: 0.866 (+/- 0.114)
f1: 0.854 (+/- 0.072)
roc_auc: 0.970 (+/- 0.017)

Random Forest
accuracy: 0.902 (+/- 0.044)
recall: 0.876 (+/- 0.103)
f1: 0.853 (+/- 0.070)
roc_auc: 0.960 (+/- 0.030)

XGBoost
accuracy: 0.915 (+/- 0.028)
recall: 0.876 (+/- 0.071)
f1: 0.871 (+/- 0.042)
roc_auc: 0.967 (+/- 0.021)


In [87]:
feature_names = model.named_steps[
    "preprocessing"
].get_feature_names_out()

coefficients = model.named_steps[
    "classifier"
].coef_[0]

importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

importance["Absolute_Coefficient"] = (
    importance["Coefficient"].abs()
)

importance.sort_values(
    "Absolute_Coefficient",
    ascending=False
).head(15)

,Feature,Coefficient,Absolute_Coefficient
14,num__G2,-4.107647,4.107647
13,num__G1,-1.756744,1.756744
32,cat__Fjob_other,-0.786182,0.786182
6,num__famrel,-0.759450,0.759450
0,num__age,0.627530,0.627530
33,cat__Fjob_services,0.601882,0.601882
31,cat__Fjob_health,0.601397,0.601397
10,num__Walc,-0.522638,0.522638
41,cat__guardian_other,0.458475,0.458475
35,cat__reason_course,0.366566,0.366566


In [88]:
X_no_grades = X.drop(columns=["G1", "G2"])

num_no_grades = X_no_grades.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

cat_no_grades = X_no_grades.select_dtypes(
    include=["object"]
).columns.tolist()

preprocessor_no_grades = ColumnTransformer([
    ("num", StandardScaler(), num_no_grades),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_no_grades)
])

model_no_grades = Pipeline([
    ("preprocessing", preprocessor_no_grades),
    ("classifier", LogisticRegression(max_iter=1000))
])

X_train_ng = X_no_grades.loc[X_train.index]
X_test_ng = X_no_grades.loc[X_test.index]

model_no_grades.fit(X_train_ng, y_train)

pred_ng = model_no_grades.predict(X_test_ng)
prob_ng = model_no_grades.predict_proba(X_test_ng)[:, 1]

print("Accuracy:", accuracy_score(y_test, pred_ng))
print("Recall:", recall_score(y_test, pred_ng))
print("F1-score:", f1_score(y_test, pred_ng))
print("ROC-AUC:", roc_auc_score(y_test, prob_ng))

C:\Users\DELL\AppData\Local\Temp\ipykernel_11164\315004238.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_no_grades = X_no_grades.select_dtypes(


Accuracy: 0.6962025316455697
Recall: 0.3076923076923077
F1-score: 0.4
ROC-AUC: 0.7148040638606676


In [89]:
cv_results_ng = cross_validate(
    model_no_grades,
    X_train_ng,
    y_train,
    cv=cv,
    scoring=["accuracy", "recall", "f1", "roc_auc"]
)

for metric in ["accuracy", "recall", "f1", "roc_auc"]:
    scores = cv_results_ng[f"test_{metric}"]
    print(f"{metric}: {scores.mean():.3f} (+/- {scores.std():.3f})")

accuracy: 0.674 (+/- 0.059)
recall: 0.338 (+/- 0.104)
f1: 0.401 (+/- 0.088)
roc_auc: 0.633 (+/- 0.047)


In [90]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, roc_auc_score
)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

print("Baseline accuracy:", accuracy_score(y_test, baseline_pred))
print("Baseline recall:", recall_score(y_test, baseline_pred))
print("Baseline F1-score:", f1_score(y_test, baseline_pred))

Baseline accuracy: 0.6708860759493671
Baseline recall: 0.0
Baseline F1-score: 0.0


In [91]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[52  1]
 [ 4 22]]


# Final Results Summary

## Research Question
How well can machine-learning models predict whether a student's final grade will be below 10, and how much do earlier academic grades contribute to prediction performance?

## Dataset
- Dataset: Student Performance — Mathematics (`student-mat.csv`)
- Records: 395 students
- Original features: 33 columns
- Target: `low_grade`, defined as 1 when G3 < 10 and 0 otherwise
- Low-grade cases: 130
- Other cases: 265

## Models
1. Logistic Regression
2. Random Forest
3. XGBoost

## Main Findings
- Logistic Regression achieved 93.67% accuracy on the held-out test set.
- XGBoost achieved the highest mean cross-validation accuracy (91.5%) and F1-score (87.1%).
- Random Forest achieved the highest test-set recall (88.46%).
- Removing G1 and G2 substantially reduced Logistic Regression performance. In five-fold cross-validation, recall decreased from 86.6% to 33.8%, and ROC-AUC decreased from 97.0% to 63.3%.
- The majority-class baseline achieved 67.09% test accuracy and zero recall for the low-grade class.

## Limitations
- The dataset contains final grades, not confirmed student dropout outcomes.
- The dataset is relatively small, with 395 records.
- Test-set metrics are based on 79 records.
- Results may vary across different samples, schools, and student populations.
- Earlier grades are strongly informative, so prediction performance depends substantially on when the prediction is intended to be made.

## Conclusion
The experiments suggest that earlier academic grades provide substantial predictive information for identifying students whose final mathematics grade is below 10. The three models show useful predictive performance on this dataset, but further validation is needed before generalizing the results. These findings concern low-final-grade prediction, not actual student dropout prediction.

In [92]:
print("Scenario 1: With earlier grades (G1 and G2)")
print("Features:", X.shape[1])
print("Cross-validation recall: 0.866")
print("Cross-validation ROC-AUC: 0.970")

print("\nScenario 2: Without earlier grades (G1 and G2)")
print("Features:", X_no_grades.shape[1])
print("Cross-validation recall: 0.338")
print("Cross-validation ROC-AUC: 0.633")

Scenario 1: With earlier grades (G1 and G2)
Features: 32
Cross-validation recall: 0.866
Cross-validation ROC-AUC: 0.970

Scenario 2: Without earlier grades (G1 and G2)
Features: 30
Cross-validation recall: 0.338
Cross-validation ROC-AUC: 0.633
